# 03 — Modeling experiments (and why there is no model)

Target K12 first (clean labels), then K11. Design, fixed before any fit:

- **Grain:** one row per order; label = 1 iff `RETURNED`.
- **Censoring:** `IN TRANSIT` outcomes are unknown — training uses resolved orders only.
- **No leakage:** customer history on strictly earlier days; category/brand rates encoded on TRAIN only; **time split** (train < 2025-10-01).
- **Metric:** PR-AUC (11.6% base rate makes accuracy lie); ROC-AUC alongside.
- **Candidates:** logistic regression, random forest, MLP(64→32). If a universal approximator can't beat base rate, nothing can.

Needs seeded PostgreSQL (`PGHOST/…`, see `scripts/seed/README.md`). Companions: `ml/local_proof.py`, `ml/churn_feasibility.py`.

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from backend.app.core.config import get_settings
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from ml.local_proof import ALPHA, FEATURE_SQL, SPLIT

engine = create_engine(get_settings().dsn)
t0 = time.time()
df = pd.read_sql(text(FEATURE_SQL), engine)
print(f"resolved orders: {len(df):,} in {time.time()-t0:.0f}s · base rate {df['is_returned'].mean():.4f}")

## Features (identical for every candidate — only the learner changes)

In [ ]:
def featurize(df, gmean=None, maps=None):
    df = df.copy()
    if maps is None:  # TRAIN: fit smoothed target rates
        maps, gmean = {}, df["is_returned"].mean()
        for col in ("category", "brand"):
            agg = df.groupby(col)["is_returned"].agg(["sum", "count"])
            maps[col] = (agg["sum"] + ALPHA * gmean) / (agg["count"] + ALPHA)
    for col in ("category", "brand"):
        df[f"{col}_rate"] = df[col].map(maps[col]).fillna(gmean)
    df["log_price"] = np.log1p(df["unit_price"].astype(float))
    df["month"] = pd.to_datetime(df["order_date"]).dt.month
    df["dow"] = pd.to_datetime(df["order_date"]).dt.dayofweek
    num = ["prior_orders", "prior_returns", "prior_return_rate", "tenure_days", "days_since_last",
           "discount_pct", "log_price", "quantity", "category_rate", "brand_rate", "month", "dow"]
    enc = pd.get_dummies(df, columns=["payment_method", "device"])
    return enc, num, maps, gmean

tr = df[df["order_date"].astype(str) < SPLIT]
te = df[df["order_date"].astype(str) >= SPLIT]
tr_enc, num, maps, gmean = featurize(tr)
te_enc, _, _, _ = featurize(te, gmean, maps)
for c in list(tr_enc.columns):
    if c.startswith(("payment_method_", "device_")) and c not in te_enc:
        te_enc[c] = 0
feats = num + [c for c in tr_enc.columns if c.startswith(("payment_method_", "device_"))]
Xtr, ytr = tr_enc[feats].astype(float).values, tr_enc["is_returned"].astype(int).values
Xte, yte = te_enc[feats].astype(float).values, te_enc["is_returned"].astype(int).values
print(f"train {len(Xtr):,} / test {len(Xte):,} · {len(feats)} features")

## Candidates — linear, forest, deep net

In [ ]:
curves = {}

for name, model, Xs in [
    ("logreg", LogisticRegression(max_iter=1000, class_weight="balanced"), (Xtr, Xte)),
    ("forest", RandomForestClassifier(n_estimators=300, min_samples_leaf=50, class_weight="balanced_subsample", n_jobs=-1, random_state=42), (Xtr, Xte)),
]:
    t0 = time.time()
    model.fit(*Xs[:1], ytr)
    p = model.predict_proba(Xs[1])[:, 1]
    curves[name] = p
    print(f"{name}: PR-AUC={average_precision_score(yte, p):.4f} ROC-AUC={roc_auc_score(yte, p):.4f} ({time.time()-t0:.0f}s)")

sc = StandardScaler().fit(Xtr)
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=30, random_state=42)
t0 = time.time()
mlp.fit(sc.transform(Xtr), ytr)
p = mlp.predict_proba(sc.transform(Xte))[:, 1]
curves["mlp(64-32)"] = p
print(f"mlp: PR-AUC={average_precision_score(yte, p):.4f} ROC-AUC={roc_auc_score(yte, p):.4f} ({time.time()-t0:.0f}s, {mlp.n_iter_} iters)")

## Same protocol, second target — K11 churn

In [ ]:
from ml.churn_feasibility import SQL as CHURN_SQL
from sklearn.model_selection import train_test_split

t0 = time.time()
dc = pd.read_sql(text(CHURN_SQL), engine)
print(f"customers: {len(dc):,} in {time.time()-t0:.0f}s · churn rate {dc['churned'].mean():.3f}")
Xc = dc[["n_orders", "tenure", "recency", "avg_gap", "revenue", "ret_rate", "avg_disc"]].fillna(-1).astype(float).values
yc = dc["churned"].astype(int).values
Xctr, Xcte, yctr, ycte = train_test_split(Xc, yc, test_size=0.25, random_state=42)
for name, model in [("logreg", LogisticRegression(max_iter=1000, class_weight="balanced")),
                      ("forest", RandomForestClassifier(n_estimators=200, min_samples_leaf=50, class_weight="balanced_subsample", n_jobs=-1, random_state=42))]:
    model.fit(Xctr, yctr); p = model.predict_proba(Xcte)[:, 1]
    if name == "forest":
        curves["churn-forest"] = None  # different test set; numbers only
    print(f"churn-{name}: PR-AUC={average_precision_score(ycte, p):.4f} ROC-AUC={roc_auc_score(ycte, p):.4f}")

## PR curves — every learner hugs the base-rate line

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
for name, p in curves.items():
    if p is None:
        continue
    prec, rec, _ = precision_recall_curve(yte, p)
    plt.plot(rec, prec, label=name, linewidth=1.5)
plt.axhline(yte.mean(), color="black", linestyle="--", label=f"base rate {yte.mean():.3f}")
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig("/tmp/opencode/nb03_pr.png"); print("plot: /tmp/opencode/nb03_pr.png")

## Verdict

| Target | Best PR-AUC | Base | ROC | Meaning |
|---|---|---|---|---|
| Return | ≈0.164 | 0.164 | ≈0.50 | chance |
| Churn | ≈0.872 | 0.872 | ≈0.50 | chance |

A universal approximator with 530k rows can't beat the base rate → the labels carry no learnable signal (statuses i.i.d., arrivals memoryless — see `02_statistics.ipynb`). **No model ships.** The exploitable structure is two rules (6-day shipping ≈2×, sub-3.0 rating ≈3× risk) — see the EDA verdict. This notebook stays as the experiment record.